# 🔧 Build: async agent runs + a scheduled automation

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 31 §31.8 (with §31.4, §31.6) · type: walkthrough*

**The promise:** you will build the platform's spine — a `/runs` endpoint that enqueues a Celery `run_agent` task and returns a `run_id` *immediately*; a worker that runs the agent loop, **checkpoints** every step so it survives a restart, and **publishes progress** the client subscribes to; plus a Beat **"morning digest"** automation that runs an agent with no human in the loop. This is where the capstone's `workers/` is born.

## 🧠 Why this matters

Everything in this chapter converges on one architectural shape: **the request layer accepts and dispatches; the worker layer does the slow, stateful work.** A thin, fast API sits in front; a fleet of background workers runs behind a queue. An agent that plans, calls tools, and reasons for two minutes must *never* run inside an HTTP request — the API enqueues the run and hands back a `run_id`, and a worker executes the loop while the client polls or subscribes (WebSocket/SSE, Ch 25) for progress.

Bolt on two more ideas you already built and this becomes production-grade. **Checkpointing** (Ch 14) makes a long run *durable* — kill the worker mid-run and it resumes from the last step instead of starting over. **Idempotency** (Ch 29, notebook `31-02`) makes the inevitable `acks_late` re-run *safe* — a resumed run reuses recorded outcomes instead of doubling its side effects. That shape — not any clever prompt — is what makes an agentic system real.

## Objectives & prereqs

**By the end you can:**
- Build the book's `run_agent` Celery task: resume on retry, loop the agent, **checkpoint** and **publish progress** each step, retry with backoff on `TransientError`.
- Wire the dispatch half: `/runs` enqueues and returns a `run_id` instantly; the client subscribes for progress — agent runs never block HTTP.
- Show **durable resume**: kill the worker mid-run, restart, and continue from the checkpoint.
- Register a Beat **"morning digest"** automation, and place n8n/Zapier-style platforms as triggers/actions.

**Prereqs:** `31-02` (idempotent tasks, recorded activities); Ch 25 (FastAPI, WebSocket/SSE) for the dispatch + subscribe path; Ch 14 (checkpointing); Ch 16 (the agent loop being run).

**Runs free & offline.** The agent loop and the digest are **mocked** (`MOCK=1` canned steps) so the whole build runs free and deterministically — no model calls, no Redis, no worker process. Celery runs **eager** (in-process); the checkpoint store and progress channel are in-memory. ⚠️ The Slack/automation side effect is **dry-run by default**, opt-in, and flagged. The `MOCK=0` + `docker compose` (api + worker + Redis) path is documented in setup.

In [ ]:
# Setup — imports, env, and the MOCK / dry-run switches.
import os
import random

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default): canned agent steps, eager Celery, in-memory checkpoints/progress — free & offline.
# MOCK=0: a short REAL agent run + a real broker/worker (docker compose: api + worker + Redis).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Side effects (the digest's 'post to Slack') are DRY-RUN unless you explicitly opt in.
# ⚠️ Setting this to '1' would perform a real post in MOCK=0 — leave it off for the lesson.
ALLOW_SIDE_EFFECTS = os.getenv("COMPANION_ALLOW_SIDE_EFFECTS", "0") == "1"

BROKER_URL = os.getenv("CELERY_BROKER_URL", "redis://localhost:6379/0")
RESULT_BACKEND = os.getenv("CELERY_RESULT_BACKEND", "redis://localhost:6379/1")

random.seed(3108)  # the mock agent's 'steps' are reproducible

from celery import Celery
from celery.schedules import crontab

app = Celery("platform", broker=BROKER_URL, backend=RESULT_BACKEND)
if MOCK:
    app.conf.task_always_eager = True
    app.conf.task_eager_propagates = True

print("MOCK:", MOCK, "| side effects:", "LIVE ⚠️" if ALLOW_SIDE_EFFECTS else "dry-run (safe)")

## The platform substrate: checkpoint store, progress channel, agent loop

Three small pieces stand in for the production services. The **checkpoint store** (a dict here; a DB table in the capstone, Ch 30) persists run state per step. The **progress channel** (a dict of lists here; Redis pub/sub or a WebSocket hub in production, Ch 25) is what the client subscribes to. And a **mock agent loop** yields canned steps so the build is deterministic and free.

In [ ]:
# --- Checkpoint store (Ch 14): durable per-run state. In-memory here; a DB table in capstone. ---
CHECKPOINTS = {}     # run_id -> {"goal": str, "last_step": int, "steps": [step_dict, ...]}
PROGRESS = {}        # run_id -> [progress_event, ...]  (the channel clients subscribe to)


class TransientError(Exception):
    """Expected-to-clear failure (network blip, throttle) — guaranteed in distributed systems."""


def load_or_create(run_id: str, goal: str) -> dict:
    """Resume from the last checkpoint if this run exists (a retry); otherwise start fresh."""
    if run_id in CHECKPOINTS:
        return CHECKPOINTS[run_id]
    state = {"goal": goal, "last_step": 0, "steps": []}
    CHECKPOINTS[run_id] = state
    return state


def checkpoint(run_id: str, step: dict):
    """Persist progress so a worker restart resumes here instead of from zero."""
    state = CHECKPOINTS[run_id]
    state["last_step"] = step["n"]
    state["steps"].append(step)


def publish_progress(run_id: str, step: dict):
    """Push a progress event onto the channel the client is subscribed to (SSE/WebSocket)."""
    PROGRESS.setdefault(run_id, []).append({"step": step["n"], "action": step["action"]})


# --- The mock agent loop (Ch 16). Canned, deterministic steps; MOCK=0 would run a real loop. ---
_CANNED_STEPS = [
    {"action": "search_docs",  "result": "42 docs since yesterday"},
    {"action": "summarize",    "result": "3 themes extracted"},
    {"action": "draft_digest", "result": "digest drafted"},
    {"action": "post_slack",   "result": "posted to #daily-digest"},
]


def agent_loop(state: dict):
    """Yield the remaining steps, RESUMING after state['last_step'] (durable resume).
    Each step carries an idempotency key so a re-run reuses, never repeats, its outcome."""
    for i, base in enumerate(_CANNED_STEPS, start=1):
        if i <= state["last_step"]:
            continue   # already done in a prior (crashed) attempt — skip on resume
        yield {"n": i, "idempotency_key": f"{state.get('run_id','')}:{i}", **base}


print("substrate ready: checkpoint store, progress channel, mock agent loop (4 steps)")

## 🔧 Build the book's `run_agent` task

Here is the chapter's task verbatim in shape: `bind=True, acks_late=True, max_retries=3`. It `load_or_create`s state (so a retry **resumes** instead of restarting), loops the agent, **checkpoints** and **publishes progress** each step, and retries with exponential backoff on a `TransientError`.

In [ ]:
@app.task(bind=True, acks_late=True, max_retries=3)
def run_agent(self, run_id: str, goal: str):
    state = load_or_create(run_id, goal)    # resume if this is a retry
    state["run_id"] = run_id
    try:
        for step in agent_loop(state):
            checkpoint(run_id, step)         # durable: survives a worker restart (Ch 14)
            publish_progress(run_id, step)   # client sees live progress (Ch 25)
        return {"run_id": run_id, "status": "done", "steps": state["last_step"]}
    except TransientError as exc:
        # exponential backoff: 1s, 2s, 4s
        raise self.retry(exc=exc, countdown=2 ** self.request.retries)


print("task registered:", run_agent.name)

## The dispatch half: `/runs` enqueues and returns *now*

The request layer's whole job is to **accept and dispatch**. `/runs` mints a `run_id`, enqueues the task with `.delay()`, and returns the id immediately — the agent run never blocks the HTTP response. The client then **subscribes** (or polls) for progress on that id. (This is a plain function here; in the capstone it's a FastAPI route, Ch 25.)

In [ ]:
import uuid


def post_runs(goal: str) -> dict:
    """POST /runs — accept a goal, dispatch a background run, return a run_id instantly."""
    run_id = f"run-{uuid.UUID(int=random.getrandbits(128)).hex[:8]}"
    run_agent.delay(run_id, goal)            # enqueue; the worker runs the loop elsewhere
    return {"run_id": run_id, "status": "accepted"}   # 202 Accepted — HTTP returns now


def get_progress(run_id: str) -> list:
    """What a subscribed client (SSE/WebSocket) would receive as the worker advances."""
    return PROGRESS.get(run_id, [])


resp = post_runs("Summarize docs added in the last 24h; post to #daily-digest.")
print("POST /runs ->", resp, "(client got a run_id without waiting for the agent)")
print("\nprogress stream the client subscribed to:")
for ev in get_progress(resp["run_id"]):
    print("   step", ev["step"], "->", ev["action"])

**What you just saw.** `/runs` returned a `run_id` and `accepted` — in eager mode the worker then drained all four steps, and the progress channel shows each one. Against a real broker, `/runs` would return *before* step 1 even starts, and the client would watch the steps arrive live over SSE/WebSocket. The HTTP tier stayed thin and fast; the slow work happened behind the queue.

## 🔮 Predict: kill the worker at step 3, then restart

Now the payoff of checkpointing. We run a fresh agent run but **crash the worker after step 2** (a `TransientError` raised at step 3). Then a retry re-invokes `run_agent` with the *same* `run_id`.

**Predict:** when it resumes, does it **re-run steps 1–2** from zero, or **continue at step 3** from the checkpoint? And how many total steps get checkpointed — 4, or 6? Decide, then run.

In [ ]:
# A version that crashes once, partway, to demonstrate durable resume.
CRASH_STATE = {"crashed": False}
RUN_ID = "run-resume-demo"


def run_agent_with_crash(run_id, goal):
    state = load_or_create(run_id, goal)
    state["run_id"] = run_id
    for step in agent_loop(state):
        if step["n"] == 3 and not CRASH_STATE["crashed"]:
            CRASH_STATE["crashed"] = True
            print(f"  💥 worker killed entering step 3 (last good checkpoint: step {state['last_step']})")
            return "crashed"
        checkpoint(run_id, step)
        publish_progress(run_id, step)
        print(f"  ✓ step {step['n']} ({step['action']}) checkpointed")
    return "done"


print("attempt 1 (will crash):")
run_agent_with_crash(RUN_ID, "daily digest")
print("checkpointed steps so far:", CHECKPOINTS[RUN_ID]["last_step"])

print("\nattempt 2 (acks_late re-delivery → retry, same run_id):")
outcome = run_agent_with_crash(RUN_ID, "daily digest")
print("outcome:", outcome, "| total checkpointed steps:", CHECKPOINTS[RUN_ID]["last_step"])
assert CHECKPOINTS[RUN_ID]["last_step"] == 4, "resume did not complete exactly 4 steps"
print("PASS — resumed at step 3, finished at 4: NO step ran twice")

**What you just saw.** The crash left checkpoints at step 2. The retry called `load_or_create` with the same `run_id`, found the saved state, and `agent_loop` **skipped steps 1–2** and resumed at step 3 — finishing at 4, with **no step executed twice**. That is durable resume: the run survived a worker death because its progress lived in the checkpoint store, not in the worker's memory.

## ⚠️ Pitfall: re-runs must be idempotent *per step*

Durable resume skips *completed* steps, but `acks_late` can still re-deliver a task whose final step finished just before the crash. So each step's **side effect** must be idempotent — keyed by `run_id:step` (Ch 29, notebook `31-02`) — or a resumed run that re-enters a half-finished step doubles its effect (a second Slack post, a second charge). Below: the `post_slack` step guarded by a per-step idempotency ledger, re-invoked to prove it stays exactly-once.

In [ ]:
STEP_LEDGER = {}     # idempotency_key -> recorded outcome (the Ch 29 ledger)
SLACK_POSTS = []      # the real-world side effect we must not double


def do_step_side_effect(step: dict):
    """Perform a step's side effect exactly once, keyed by run_id:step."""
    key = step["idempotency_key"]
    if key in STEP_LEDGER:
        return {**STEP_LEDGER[key], "replayed": True}   # resumed into a finished step → no-op
    SLACK_POSTS.append(step["result"])                   # the side effect, once
    STEP_LEDGER[key] = {"posted": step["result"]}
    return {**STEP_LEDGER[key], "replayed": False}


post_step = {"n": 4, "action": "post_slack", "result": "posted to #daily-digest",
             "idempotency_key": "run-resume-demo:4"}

first = do_step_side_effect(post_step)
second = do_step_side_effect(post_step)   # acks_late re-delivery of the SAME step

print("first :", first)
print("second:", second)
print("slack posts:", len(SLACK_POSTS), "(1 → idempotent per step; the re-run was a no-op)")
assert len(SLACK_POSTS) == 1, "side effect doubled — step was not idempotent"

## Scheduled automation: the "morning digest"

The platform doing useful work with **no human in the loop**. A Beat job each morning runs a digest agent over the last 24h of docs and posts a summary. Same `run_agent` task, fired by the scheduler instead of an HTTP request. The Slack post is **dry-run** unless you explicitly opt in (`COMPANION_ALLOW_SIDE_EFFECTS=1`).

In [ ]:
app.conf.beat_schedule = {
    "morning-digest": {
        "task": run_agent.name,                         # 'platform.run_agent'
        "schedule": crontab(hour=7, minute=0),          # 07:00 every day
        "args": ("digest", "Summarize docs added in the last 24h; post to Slack."),
    },
}
entry = app.conf.beat_schedule["morning-digest"]
print("Beat 'morning-digest' registered:")
print("  task    :", entry["task"])
print("  schedule:", entry["schedule"], "(07:00 daily)")
print("  args    :", entry["args"])


def post_to_slack(text: str):
    """⚠️ Side effect — DRY-RUN by default. Opt in with COMPANION_ALLOW_SIDE_EFFECTS=1."""
    if not ALLOW_SIDE_EFFECTS:
        print(f"  [dry-run] would post to Slack: {text!r}")
        return {"dry_run": True}
    # Live path (MOCK=0 + opt-in): a real client reads SLACK_WEBHOOK_URL from the env.
    import httpx

    url = os.environ["SLACK_WEBHOOK_URL"]   # fails fast & friendly if missing
    httpx.post(url, json={"text": text}, timeout=10)
    return {"dry_run": False}


print("\nsimulating the 07:00 fire:")
post_to_slack("Daily digest: 3 themes from 42 new docs.")

## Integration platforms as triggers & actions

Not every automation should be code. **n8n** (open-source, self-hostable), Zapier, or Make connect SaaS tools with visual workflows in two directions that matter for an agentic platform: as **triggers** that kick off your agents from external events ("a row was added → call the digest agent"), and as **actions** your agents reach — often the *same* tools you already expose over **MCP** (Ch 19).

In [ ]:
# Two directions an integration platform plugs into your agents.
INTEGRATION = {
    "trigger": "external event (new sheet row, inbound email, webhook) → POST /runs → an agent run",
    "action": "an agent's tool call → n8n/Zapier node → a SaaS effect (often the same tool over MCP)",
}
for direction, desc in INTEGRATION.items():
    print(f"{direction:8}: {desc}")

# A 'trigger' from an integration platform is just another caller of post_runs():
triggered = post_runs("New contract uploaded — extract key terms and flag risks.")
print("\nn8n trigger dispatched a run:", triggered["run_id"], "->", triggered["status"])

## 🎯 Senior lens

Internalize this shape, because it answers nearly every "how do I run agents in production?" question: **the request layer accepts and dispatches; the worker layer does the slow, stateful work.** A thin, fast API in front; a fleet of background workers behind a queue, each running durable, idempotent agent tasks. It keeps the user-facing tier responsive *and* lets you scale the expensive agent work independently — you add capacity by simply adding workers, no code change.

And know when *not* to write code at all: the long tail of glue — "when X happens, run the agent and post the result" — often belongs in an n8n/Zapier automation that ops can own and edit, not a bespoke service only you can maintain. Custom code for the differentiated core; a visual platform for the integrations. This architecture, not any single clever prompt, is what makes an agentic system production-grade — and it's the version the capstone deploys on AWS in Part VIII.

## Recap

- **The spine:** request layer **accepts & dispatches** (`/runs` enqueues, returns a `run_id` instantly); worker layer **does the slow work** behind a queue. Agent runs never block HTTP.
- **`run_agent`** (`bind=True, acks_late=True, max_retries=3`): `load_or_create` to **resume on retry**, loop the agent, **`checkpoint`** + **`publish_progress`** each step, `self.retry` with backoff on `TransientError`.
- **Checkpointing (Ch 14) → durable resume:** kill the worker mid-run, restart, continue from the last step — no step re-runs.
- **Idempotency per step (Ch 29):** keyed by `run_id:step`, so an `acks_late` re-entry into a finished step is a no-op — side effects stay exactly-once.
- **Beat `morning-digest`** (`crontab(hour=7, minute=0)`) runs an agent autonomously; the Slack post is **dry-run by default**, opt-in, ⚠️-flagged.
- **Integration platforms** (n8n/Zapier/Make) are **triggers** into and **actions** out of your agents — often the same tools exposed over MCP (Ch 19).
- **Scale by adding workers.** This shape — not a clever prompt — is what makes an agentic system production-grade.

## Exercises

Each *changes something* and asks you to predict the result first.

1. **Crash at a different step.** Change the crash point to step 2 (instead of 3). Predict how many steps are checkpointed before and after the retry, then verify the run still finishes at 4 with no step run twice.
2. **Add a sixth step.** Append a `"notify_owner"` step to `_CANNED_STEPS`. Predict the new progress stream and the final `last_step`, then run a fresh `post_runs(...)` and confirm.
3. **Prove the no-op on full re-delivery.** Re-invoke `run_agent.delay(RUN_ID, ...)` for a run that already completed (all steps checkpointed). Predict how many *new* progress events and Slack posts occur, then verify it's zero.
4. **Second schedule.** Register an `evening-rollup` Beat job at `crontab(hour=18, minute=30)` pointing at `run_agent` with a different goal. Predict how it differs operationally from `morning-digest`, then assert the `crontab` matches 18:30.

In [ ]:
# Exercise 1 — your code here.


In [ ]:
# Exercise 2 — your code here.


In [ ]:
# Exercise 3 — your code here.


## Next

- **Blueprint this built (the real version):** [`blueprints/agent-loop/`](../../../blueprints/agent-loop/) — the production `run_agent` wraps this loop, idempotent, retried, and checkpointed. You built the toy; that's the real one.
- **Template:** [`templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/) — its `docker compose` (api + worker + Redis) is the `MOCK=0` substrate for everything here.
- **Capstone:** this **builds `capstone/workers/`** (the Celery app, `run_agent`, Beat schedules) and the async `/runs` dispatch in `capstone/app/`; checkpoint `checkpoints/ch31-workers-and-automation`. Deployed on AWS in **Part VIII**.
- **Book:** §31.8 (this Build), §31.4 (agent runs in the background), §31.6 (integration/automation platforms).